**Read Bronze**

In [0]:
bronze_df = spark.table("ecommerce.bronze.orders")

display(bronze_df)

order_id,order_date,customer_name,product,category,quantity,price
1001,2026-09-01,Ravi,Laptop,Electronics,1,800
1002,2026-09-01,John,Mouse,Electronics,2,25
1003,2026-09-01,Priya,Shoes,Fashion,1,60
1004,2026-09-01,David,Keyboard,Electronics,1,45
1005,2026-09-01,Anitha,T-Shirt,Fashion,3,20
1006,2026-09-01,Robert,Headphones,Electronics,1,75
1007,2026-09-01,Suresh,Jeans,Fashion,2,40
1008,2026-09-01,Emily,Coffee Maker,Home,1,90
1009,2026-09-01,Arun,Monitor,Electronics,1,250
1010,2026-09-01,Sarah,Backpack,Fashion,1,35


**Schema**

In [0]:
bronze_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



**First data-quality check - Missing Order IDs**

In [0]:
from pyspark.sql.functions import col

bronze_df.filter(
    col("order_id").isNull()
).show()

+--------+----------+-------------+-------+--------+--------+-----+
|order_id|order_date|customer_name|product|category|quantity|price|
+--------+----------+-------------+-------+--------+--------+-----+
+--------+----------+-------------+-------+--------+--------+-----+



**Check for invalid quantities**

In [0]:
bronze_df.filter(
    col("quantity").isNull()
).show()

+--------+----------+-------------+-------+--------+--------+-----+
|order_id|order_date|customer_name|product|category|quantity|price|
+--------+----------+-------------+-------+--------+--------+-----+
+--------+----------+-------------+-------+--------+--------+-----+



**Check for zero or negative quantities**

In [0]:
bronze_df.filter(
    col("quantity") <= 0
).show()

+--------+----------+-------------+-------+-----------+--------+-----+
|order_id|order_date|customer_name|product|   category|quantity|price|
+--------+----------+-------------+-------+-----------+--------+-----+
|    1041|2026-09-04|    Test User|  Mouse|Electronics|      -2|   25|
+--------+----------+-------------+-------+-----------+--------+-----+



**Clean the data**

In [0]:
silver_df = bronze_df.filter(
    col("order_id").isNotNull()
)

silver_df = silver_df.filter(
    col("quantity") > 0
)

display(silver_df)

order_id,order_date,customer_name,product,category,quantity,price
1001,2026-09-01,Ravi,Laptop,Electronics,1,800
1002,2026-09-01,John,Mouse,Electronics,2,25
1003,2026-09-01,Priya,Shoes,Fashion,1,60
1004,2026-09-01,David,Keyboard,Electronics,1,45
1005,2026-09-01,Anitha,T-Shirt,Fashion,3,20
1006,2026-09-01,Robert,Headphones,Electronics,1,75
1007,2026-09-01,Suresh,Jeans,Fashion,2,40
1008,2026-09-01,Emily,Coffee Maker,Home,1,90
1009,2026-09-01,Arun,Monitor,Electronics,1,250
1010,2026-09-01,Sarah,Backpack,Fashion,1,35


**Calculate Revenue**

In [0]:
silver_df = silver_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

display(silver_df)

order_id,order_date,customer_name,product,category,quantity,price,revenue
1001,2026-09-01,Ravi,Laptop,Electronics,1,800,800
1002,2026-09-01,John,Mouse,Electronics,2,25,50
1003,2026-09-01,Priya,Shoes,Fashion,1,60,60
1004,2026-09-01,David,Keyboard,Electronics,1,45,45
1005,2026-09-01,Anitha,T-Shirt,Fashion,3,20,60
1006,2026-09-01,Robert,Headphones,Electronics,1,75,75
1007,2026-09-01,Suresh,Jeans,Fashion,2,40,80
1008,2026-09-01,Emily,Coffee Maker,Home,1,90,90
1009,2026-09-01,Arun,Monitor,Electronics,1,250,250
1010,2026-09-01,Sarah,Backpack,Fashion,1,35,35


**Compare the bronze and Silver count**

In [0]:
print("Bronze record count:", bronze_df.count())
print("Silver record count:", silver_df.count())

Bronze record count: 41
Silver record count: 40


**Save Silver as a Delta table**

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.silver.orders")

**Verify the Silver table**

In [0]:
silver_table = spark.table("ecommerce.silver.orders")

display(silver_table)

order_id,order_date,customer_name,product,category,quantity,price,revenue
1001,2026-09-01,Ravi,Laptop,Electronics,1,800,800
1002,2026-09-01,John,Mouse,Electronics,2,25,50
1003,2026-09-01,Priya,Shoes,Fashion,1,60,60
1004,2026-09-01,David,Keyboard,Electronics,1,45,45
1005,2026-09-01,Anitha,T-Shirt,Fashion,3,20,60
1006,2026-09-01,Robert,Headphones,Electronics,1,75,75
1007,2026-09-01,Suresh,Jeans,Fashion,2,40,80
1008,2026-09-01,Emily,Coffee Maker,Home,1,90,90
1009,2026-09-01,Arun,Monitor,Electronics,1,250,250
1010,2026-09-01,Sarah,Backpack,Fashion,1,35,35


**One final validation**

In [0]:
print("Silver record count:", silver_table.count())

Silver record count: 40
